# Wikidata VGAE Training

Train the R-GCN-based variational graph autoencoder on the same `tkgl-smallpedia` Wikidata graph used by the R-GCN experiments. The notebook loads `R-GCN/checkpoints/wiki_graph.pkl` and saves training results to a pickle file.

In [1]:
from pathlib import Path
import datetime as dt
import importlib
import pickle
import random
import sys

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "R-GCN" / "rgcn_model.py").exists():
            return candidate
    raise RuntimeError("Could not find project root containing R-GCN/rgcn_model.py")


PROJECT_ROOT = find_project_root(Path.cwd())
RGCN_DIR = PROJECT_ROOT / "R-GCN"
CHECKPOINT_DIR = RGCN_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility for wiki_graph.pkl files saved from notebooks.
RelationalGraph = type("RelationalGraph", (), {})


class GraphCheckpointUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__" and name == "RelationalGraph":
            return RelationalGraph
        return super().find_class(module, name)

sys.path.insert(0, str(RGCN_DIR))

import vgae_model
vgae_model = importlib.reload(vgae_model)

VGAE = vgae_model.VGAE
add_reciprocal_edges = vgae_model.add_reciprocal_edges
build_known_edge_ids = vgae_model.build_known_edge_ids
sample_vgae_negatives = vgae_model.sample_vgae_negatives
sample_vgae_tail_negatives = vgae_model.sample_vgae_tail_negatives
kl_divergence = vgae_model.kl_divergence
kl_beta_for_epoch = vgae_model.kl_beta_for_epoch

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

/cluster/tufts/apps/manual/9/x86_64/jupyter/4.5.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /cluster/home/amumma01/290-Final-Project
Device: cuda


In [2]:
GRAPH_CKPT = RGCN_DIR / "checkpoints" / "wiki_graph.pkl"
RESULTS_PATH = CHECKPOINT_DIR / "vgae_wikidata_results.pkl"

if not GRAPH_CKPT.exists():
    raise FileNotFoundError(
        f"Missing {GRAPH_CKPT}. Run R-GCN/wikidata.ipynb first to build wiki_graph.pkl."
    )

with GRAPH_CKPT.open("rb") as f:
    wiki_graph = GraphCheckpointUnpickler(f).load()

print(f"Loaded graph: {wiki_graph.name}")
print(f"Nodes: {wiki_graph.num_nodes:,}")
print(f"Relations: {wiki_graph.num_relations:,}")
print(f"Train edges: {len(wiki_graph.train_edges):,}")
print(f"Val edges: {len(wiki_graph.val_edges):,}")
print(f"Test edges: {len(wiki_graph.test_edges):,}")
print(f"Node features: {wiki_graph.node_features is not None}")

Loaded graph: tkgl-smallpedia
Nodes: 47,433
Relations: 566
Train edges: 775,514
Val edges: 162,066
Test edges: 163,172
Node features: False


In [3]:
SEED = 42
HIDDEN_DIM = 64
LATENT_DIM = 64
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 4096
NEG_RATIO = 3
KL_BETA = 1e-4
KL_WARMUP_EPOCHS = 10
KL_PRETRAIN_EPOCHS = 5
WEIGHT_DECAY = 1e-5
USE_ADAMW = True
USE_RECIPROCAL_EDGES = False
FILTER_TRAIN_NEGATIVES = False
FILTER_EVAL_NEGATIVES = False
EVAL_EVERY = 5
EVAL_POS_LIMIT = 2000
EVAL_NEGATIVES = 50
EVAL_BATCH_SIZE = 4096

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

config = {
    "seed": SEED,
    "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM,
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "neg_ratio": NEG_RATIO,
    "kl_beta": KL_BETA,
    "kl_warmup_epochs": KL_WARMUP_EPOCHS,
    "kl_pretrain_epochs": KL_PRETRAIN_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "use_adamw": USE_ADAMW,
    "use_reciprocal_edges": USE_RECIPROCAL_EDGES,
    "filter_train_negatives": FILTER_TRAIN_NEGATIVES,
    "filter_eval_negatives": FILTER_EVAL_NEGATIVES,
    "eval_every": EVAL_EVERY,
    "eval_pos_limit": EVAL_POS_LIMIT,
    "eval_negatives": EVAL_NEGATIVES,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "device": str(DEVICE),
}
EFFECTIVE_NUM_RELATIONS = wiki_graph.num_relations * (2 if USE_RECIPROCAL_EDGES else 1)
config["effective_num_relations"] = EFFECTIVE_NUM_RELATIONS
config

{'seed': 42,
 'hidden_dim': 64,
 'latent_dim': 64,
 'epochs': 50,
 'lr': 0.001,
 'batch_size': 4096,
 'neg_ratio': 3,
 'kl_beta': 0.0001,
 'kl_warmup_epochs': 10,
 'kl_pretrain_epochs': 5,
 'weight_decay': 1e-05,
 'use_adamw': True,
 'use_reciprocal_edges': False,
 'filter_train_negatives': False,
 'filter_eval_negatives': False,
 'eval_every': 5,
 'eval_pos_limit': 2000,
 'eval_negatives': 50,
 'eval_batch_size': 4096,
 'device': 'cuda',
 'effective_num_relations': 566}

In [4]:
@torch.no_grad()
def evaluate_vgae(
    model,
    graph,
    split_edges,
    edge_index,
    edge_type,
    *,
    num_negatives=50,
    max_pos=2000,
    eval_batch_size=4096,
    known_edge_ids=None,
    num_relations=None,
):
    model.eval()
    split_edges = split_edges.to(DEVICE)
    if max_pos is not None and len(split_edges) > max_pos:
        idx = torch.randperm(len(split_edges), device=DEVICE)[:max_pos]
        split_edges = split_edges[idx]

    node_features = graph.node_features.to(DEVICE) if graph.node_features is not None else None
    z, _, _ = model(edge_index, edge_type, node_features=node_features, num_nodes=graph.num_nodes)

    reciprocal_ranks = []
    hits_at_1 = []
    hits_at_3 = []
    hits_at_10 = []

    for start in range(0, len(split_edges), eval_batch_size):
        batch = split_edges[start : start + eval_batch_size]
        src = batch[:, 0]
        dst = batch[:, 1]
        rel = batch[:, 2]
        pos_scores = model.decoder(z, src, dst, rel)

        neg_dst = sample_vgae_tail_negatives(
            src,
            rel,
            graph.num_nodes,
            num_negatives,
            known_edge_ids=known_edge_ids,
            num_relations=num_relations,
        )
        neg_scores = model.decoder(
            z,
            src[:, None].expand(-1, num_negatives).reshape(-1),
            neg_dst.reshape(-1),
            rel[:, None].expand(-1, num_negatives).reshape(-1),
        ).view(len(batch), num_negatives)

        ranks = ((neg_scores >= pos_scores[:, None]).sum(dim=1) + 1).float()
        reciprocal_ranks.append(1.0 / ranks)
        hits_at_1.append((ranks <= 1).float())
        hits_at_3.append((ranks <= 3).float())
        hits_at_10.append((ranks <= 10).float())

    reciprocal_ranks = torch.cat(reciprocal_ranks)
    hits_at_1 = torch.cat(hits_at_1)
    hits_at_3 = torch.cat(hits_at_3)
    hits_at_10 = torch.cat(hits_at_10)
    return {
        "num_pos": int(len(split_edges)),
        "mrr": float(reciprocal_ranks.mean().item()),
        "hits@1": float(hits_at_1.mean().item()),
        "hits@3": float(hits_at_3.mean().item()),
        "hits@10": float(hits_at_10.mean().item()),
    }


def train_vgae(model, graph):
    model.to(DEVICE)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) if USE_ADAMW else torch.optim.Adam(model.parameters(), lr=LR)

    base_num_relations = graph.num_relations
    train_edges = graph.train_edges.to(DEVICE)
    train_edges = add_reciprocal_edges(train_edges, base_num_relations) if USE_RECIPROCAL_EDGES else train_edges
    effective_num_relations = base_num_relations * (2 if USE_RECIPROCAL_EDGES else 1)
    known_edge_ids = None
    if FILTER_TRAIN_NEGATIVES or FILTER_EVAL_NEGATIVES:
        known_edge_sets = [graph.train_edges.to(DEVICE), graph.val_edges.to(DEVICE), graph.test_edges.to(DEVICE)]
        if USE_RECIPROCAL_EDGES:
            known_edge_sets = [add_reciprocal_edges(edges, base_num_relations) for edges in known_edge_sets]
        known_edge_ids = build_known_edge_ids(known_edge_sets, graph.num_nodes, effective_num_relations)
    edge_index = train_edges[:, :2].t().contiguous()
    edge_type = train_edges[:, 2]
    node_features = graph.node_features.to(DEVICE) if graph.node_features is not None else None

    history = {"loss": [], "bce_loss": [], "kl_loss": [], "kl_beta": [], "pos_logit": [], "neg_logit": [], "mu_abs": [], "val": []}

    for epoch in tqdm(range(1, EPOCHS + 1), desc="Training VGAE"):
        model.train()
        shuffled_edges = train_edges[torch.randperm(len(train_edges), device=DEVICE)]
        epoch_loss = 0.0
        epoch_bce = 0.0
        epoch_kl = 0.0
        epoch_pos_logit = 0.0
        epoch_neg_logit = 0.0
        epoch_mu_abs = 0.0
        num_batches = 0

        for start in range(0, len(shuffled_edges), BATCH_SIZE):
            pos_edges = shuffled_edges[start : start + BATCH_SIZE]
            neg_edges = sample_vgae_negatives(
                pos_edges,
                graph.num_nodes,
                NEG_RATIO,
                known_edge_ids=known_edge_ids if FILTER_TRAIN_NEGATIVES else None,
                num_relations=effective_num_relations,
            )
            all_edges = torch.cat([pos_edges, neg_edges], dim=0)
            labels = torch.cat([
                torch.ones(len(pos_edges), device=DEVICE),
                torch.zeros(len(neg_edges), device=DEVICE),
            ])

            z, mu, log_var = model(
                edge_index,
                edge_type,
                node_features=node_features,
                num_nodes=graph.num_nodes,
            )

            logits = model.decoder(z, all_edges[:, 0], all_edges[:, 1], all_edges[:, 2])
            bce_loss = F.binary_cross_entropy_with_logits(
                logits, labels, pos_weight=labels.new_tensor(NEG_RATIO)
            )
            active_nodes = all_edges[:, :2].reshape(-1)
            kl_loss = kl_divergence(mu, log_var, active_nodes)
            effective_kl_beta = kl_beta_for_epoch(
                epoch, KL_BETA, KL_WARMUP_EPOCHS, KL_PRETRAIN_EPOCHS
            )
            loss = bce_loss + effective_kl_beta * kl_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())
            epoch_bce += float(bce_loss.item())
            epoch_kl += float(kl_loss.item())
            epoch_pos_logit += float(logits[: len(pos_edges)].mean().item())
            epoch_neg_logit += float(logits[len(pos_edges) :].mean().item())
            epoch_mu_abs += float(mu[torch.unique(active_nodes)].abs().mean().item())
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        avg_bce = epoch_bce / num_batches
        avg_kl = epoch_kl / num_batches
        history["loss"].append(avg_loss)
        history["bce_loss"].append(avg_bce)
        history["kl_loss"].append(avg_kl)
        history["kl_beta"].append(kl_beta_for_epoch(epoch, KL_BETA, KL_WARMUP_EPOCHS, KL_PRETRAIN_EPOCHS))
        history["pos_logit"].append(epoch_pos_logit / num_batches)
        history["neg_logit"].append(epoch_neg_logit / num_batches)
        history["mu_abs"].append(epoch_mu_abs / num_batches)

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            val_metrics = evaluate_vgae(
                model,
                graph,
                graph.val_edges,
                edge_index,
                edge_type,
                num_negatives=EVAL_NEGATIVES,
                max_pos=EVAL_POS_LIMIT,
                eval_batch_size=EVAL_BATCH_SIZE,
                known_edge_ids=known_edge_ids if FILTER_EVAL_NEGATIVES else None,
                num_relations=effective_num_relations,
            )
            val_metrics["epoch"] = epoch
            history["val"].append(val_metrics)
            print(
                f"Epoch {epoch:03d} | loss={avg_loss:.4f} "
                f"bce={avg_bce:.4f} kl={avg_kl:.4f} beta={history['kl_beta'][-1]:.2e} "
                f"pos={history['pos_logit'][-1]:.3f} neg={history['neg_logit'][-1]:.3f} "
                f"mu={history['mu_abs'][-1]:.3f} "
                f"val_mrr={val_metrics['mrr']:.4f} hits@10={val_metrics['hits@10']:.4f}"
            )
        else:
            print(
                f"Epoch {epoch:03d} | loss={avg_loss:.4f} "
                f"bce={avg_bce:.4f} kl={avg_kl:.4f} beta={history['kl_beta'][-1]:.2e} "
                f"pos={history['pos_logit'][-1]:.3f} neg={history['neg_logit'][-1]:.3f} "
                f"mu={history['mu_abs'][-1]:.3f}"
            )

    return history, edge_index, edge_type, known_edge_ids, effective_num_relations

In [ ]:
feat_dim = wiki_graph.node_features.shape[1] if wiki_graph.node_features is not None else None

model = VGAE(
    num_nodes=wiki_graph.num_nodes,
    hidden_dim=HIDDEN_DIM,
    num_relations=EFFECTIVE_NUM_RELATIONS,
    latent_dim=LATENT_DIM,
    feat_dim=feat_dim,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
history, train_edge_index, train_edge_type, known_edge_ids, effective_num_relations = train_vgae(model, wiki_graph)

Model parameters: 3,368,168


Training VGAE:   2%|▏         | 1/50 [01:00<49:01, 60.04s/it]

Epoch 001 | loss=0.4707 bce=0.4707 kl=227.7593 beta=0.00e+00 pos=2.745 neg=-4.337 mu=1.480


Training VGAE:   4%|▍         | 2/50 [01:58<47:05, 58.86s/it]

Epoch 002 | loss=0.1349 bce=0.1349 kl=452.5502 beta=0.00e+00 pos=4.392 neg=-9.297 mu=2.272


...

Epoch 047 | loss=0.0248 bce=0.0167 kl=81.7846 beta=1.00e-04 pos=8.586 neg=-13.665 mu=0.784


Training VGAE:  96%|█████████▌| 48/50 [46:30<01:56, 58.29s/it]

Epoch 048 | loss=0.0247 bce=0.0166 kl=81.0185 beta=1.00e-04 pos=8.596 neg=-13.732 mu=0.778


Training VGAE:  98%|█████████▊| 49/50 [47:28<00:58, 58.34s/it]

Epoch 049 | loss=0.0244 bce=0.0163 kl=80.1885 beta=1.00e-04 pos=8.638 neg=-13.766 mu=0.772


In [ ]:
test_metrics = evaluate_vgae(
    model,
    wiki_graph,
    wiki_graph.test_edges,
    train_edge_index,
    train_edge_type,
    num_negatives=EVAL_NEGATIVES,
    max_pos=EVAL_POS_LIMIT,
    eval_batch_size=EVAL_BATCH_SIZE,
    known_edge_ids=known_edge_ids if FILTER_EVAL_NEGATIVES else None,
    num_relations=effective_num_relations,
)

test_metrics

In [ ]:
model_cpu_state = {
    key: value.detach().cpu()
    for key, value in model.state_dict().items()
}

results = {
    "created_at": dt.datetime.now().isoformat(timespec="seconds"),
    "graph_checkpoint": str(GRAPH_CKPT),
    "graph": {
        "name": wiki_graph.name,
        "num_nodes": wiki_graph.num_nodes,
        "num_relations": wiki_graph.num_relations,
        "num_train_edges": int(len(wiki_graph.train_edges)),
        "num_val_edges": int(len(wiki_graph.val_edges)),
        "num_test_edges": int(len(wiki_graph.test_edges)),
        "has_node_features": wiki_graph.node_features is not None,
    },
    "config": config,
    "history": history,
    "test_metrics": test_metrics,
    "model_state_dict": model_cpu_state,
}

with RESULTS_PATH.open("wb") as f:
    pickle.dump(results, f)

print(f"Saved VGAE training results to {RESULTS_PATH}")